# 第18章 行列操作与类型转换

系统掌握新增、修改、删除、重命名和类型转换。


## 先解决一个小问题

拿一组小型业务数据练习“行列操作与类型转换”：先看数据结构，再完成一次明确的计算或转换。系统掌握新增、修改、删除、重命名和类型转换。


## 这章为什么先学

这是“Pandas”路线中第 18 章的操作重点。本章只解决“行列操作与类型转换”，不重复前面章节已经完成的准备工作。


## 开始前确认

- 掌握 Python 基础语法、列表和字典
- 开始前先确认：新增派生列


## 做完要留下什么

产出一个与“行列操作与类型转换”直接对应的结果，并记录输入形状、字段或筛选口径。


## 运行规则

代码单元格按依赖顺序执行；需要复现结果时从上到下运行，并保留输入、计算和输出。


## 本章要会

- 新增派生列
- 安全更新数据
- 删除和重命名行列
- 转换数值与分类类型


## 核心概念

- 派生列应记录计算口径。
- 链式赋值可能只修改临时对象，优先使用loc。
- 类型转换失败时应选择报错或转为缺失值。


## 示例 1：新增与修改列

assign适合链式生成新表，loc适合条件更新。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "amount": [320, 880, 460, 1250],
    "quantity": [2, 4, 1, 5],
    "status": ["完成", "完成", "取消", "完成"],
})
orders = orders.assign(unit_price=orders["amount"] / orders["quantity"])
orders.loc[orders["status"] == "取消", "amount"] = 0
print(orders)


## 示例 2：删除与重命名

drop默认返回新对象，inplace并非总是更清晰。


In [ ]:
clean = orders.drop(columns=["status"]).rename(columns={
    "amount": "sales_amount",
    "quantity": "item_count",
})
clean = clean.drop(index=2).reset_index(drop=True)
print(clean)


## 示例 3：类型转换

to_numeric的errors参数决定无效值处理方式。


In [ ]:
raw = pd.DataFrame({
    "amount": ["320.5", "N/A", "880"],
    "region": ["华东", "华南", "华东"],
})
raw["amount"] = pd.to_numeric(raw["amount"], errors="coerce")
raw["region"] = raw["region"].astype("category")
print(raw)
print(raw.dtypes)


## 公开大型数据实战

下面使用 UCI Machine Learning Repository 的 Online Retail 公开数据集。原始数据包含 541,909 条英国在线零售交易，本课程使用固定随机种子抽取的 200,000 行子集。分析时在完整子集上计算，只展示摘要或少量样本。


In [ ]:
import numpy as np
import pandas as pd
from js import window

# UCI Machine Learning Repository: Online Retail
# 原始数据 541,909 行；课程使用固定随机种子抽取的 200,000 行子集。
data_url = f"{window.location.origin}/datasets/uci_online_retail_200k.csv"
large_orders = pd.read_csv(
    data_url,
    parse_dates=["InvoiceDate"],
    dtype={"InvoiceNo": "string", "StockCode": "string", "Description": "string", "Country": "category"},
).rename(columns={
    "InvoiceNo": "order_id", "StockCode": "stock_code", "Description": "description",
    "Quantity": "quantity", "InvoiceDate": "order_time", "UnitPrice": "unit_price",
    "CustomerID": "customer_id", "Country": "country",
})
large_orders["sales"] = (large_orders["quantity"] * large_orders["unit_price"]).round(2)
large_orders["status"] = np.where(
    large_orders["order_id"].str.startswith("C") | (large_orders["quantity"] < 0),
    "取消/退货", "完成"
)
print(f"UCI Online Retail 公开数据：{len(large_orders):,} 行 × {large_orders.shape[1]} 列")
print("内存占用：", f"{large_orders.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
large_orders.head()


In [ ]:
optimized = large_orders.copy()
before_mb = optimized.memory_usage(deep=True).sum() / 1024**2
for column in ["country", "status"]:
    optimized[column] = optimized[column].astype("category")
after_mb = optimized.memory_usage(deep=True).sum() / 1024**2
print(f"类型优化前：{before_mb:.1f} MB，优化后：{after_mb:.1f} MB，节省 {(1-after_mb/before_mb):.1%}")
print(optimized.dtypes)


## 常见误区

- 触发SettingWithCopyWarning仍继续运行
- 直接覆盖原始列却没有保留转换前数据
- errors='coerce'后不检查新增缺失值


## 综合练习

1. 创建销售额和成本列
2. 计算利润和利润率
3. 把地区转换为分类类型

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“创建销售额和成本列”。
2. **独立完成**：不复制示例代码，完成“计算利润和利润率”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“把地区转换为分类类型”，用一两句话说明你修改了什么。

### 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import pandas as pd

finance = pd.DataFrame({
    "region": ["华东", "华南", "华北"],
    "sales": [1280, 960, 1100],
    "cost": [820, 710, 760],
})

# TODO: 计算利润
finance["profit"] =

# TODO: 计算利润率
finance["margin"] =

# TODO: 把地区转换为分类类型
finance["region"] =

print(finance)
print(finance.dtypes)


In [ ]:
import pandas as pd

finance = pd.DataFrame({
    "region": ["华东", "华南", "华北"],
    "sales": [1280, 960, 1100],
    "cost": [820, 710, 760],
})
finance["profit"] = finance["sales"] - finance["cost"]
finance["margin"] = finance["profit"] / finance["sales"]
finance["region"] = finance["region"].astype("category")
print(finance)
print(finance.dtypes)

# 自检
assert finance["profit"].tolist() == [460, 250, 340], "检查利润计算"
assert finance["region"].dtype.name == "category", "检查地区类型：应该是 category"


## 本章小结

系统掌握新增、修改、删除、重命名和类型转换。

**迁移思考**：

1. 如果需要批量修改多个条件下的值（如取消订单的金额改为0，退款订单的金额改为负数），应该如何组织代码？
2. 为什么使用 errors='coerce' 后要检查新增的缺失值？这些缺失值代表什么？


### 你已经掌握

- 新增派生列
- 安全更新数据
- 删除和重命名行列
- 转换数值与分类类型


### 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 新增与修改列 | assign适合链式生成新表，loc适合条件更新。 | `pd.DataFrame()`、`orders.assign()`、`orders["amount"]`、`orders["quantity"]` |
| 删除与重命名 | drop默认返回新对象，inplace并非总是更清晰。 | `orders.drop()`、`clean.drop()`、`.rename()`、`.reset_index()` |
| 类型转换 | to_numeric的errors参数决定无效值处理方式。 | `pd.DataFrame()`、`pd.to_numeric()`、`.astype()`、`raw["amount"]` |


### 需要注意

- 触发SettingWithCopyWarning仍继续运行
- 直接覆盖原始列却没有保留转换前数据
- errors='coerce'后不检查新增缺失值


### 完成检查

- [ ] 能够新增派生列
- [ ] 能够安全更新数据
- [ ] 能够删除和重命名行列
- [ ] 能够转换数值与分类类型


### 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
